In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install git+https://github.com/openai/whisper.git
!pip install --upgrade jiwer evaluate

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-azd7ocyi
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-azd7ocyi
  Resolved https://github.com/openai/whisper.git to commit ba3f3cd54b0e5b8ce1ab3de13e32122d0d5f98ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-

In [3]:
import sys
sys.path += ['/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling']

In [5]:
import train_whisper_sl
import torch, json, random, os, time
import numpy as np
import whisper
import pandas as pd

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")

path = '/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/results/'

# medium일 때 a100 38.2gb까지 사용
# 다시 시작할 때, start_epoch, ver_start, baseline_mode = False로 바꾸기

start_epoch = 0 # 끊겼을 때 다시 시작할 에폭
ver_start = 0 # 끊겼을 때 다시 시작할 버전

# meta_data = {
#     'model_name' : "openai/whisper-base",
#     'path' : path,
#     'base_model_path' : 'whisper_base_sl_baseline.pt',
#     'save_model_path' : 'whisper_sl_ver_{}.pt',
#     'save_logging_file_name' : 'logging_sl.json',
#     'epochs' : 20,
#     'start_epoch': start_epoch,
#     'ver_start': ver_start,
#     'data_size' : 1000,
#     'batch_size' : 2,
#     'lr' : 2e-5,
#     'max_len' : 100,
#     'noise_type' : 'mix1',
#     'device' : str(device)
# }

meta_data = {
    'model_name' : "openai/whisper-medium",
    'path' : path,
    'base_model_path' : 'whisper_medium_sl_baseline_50.pt',
    'save_model_path' : 'whisper_medium_sl_ver_{}_50.pt',
    'save_logging_file_name' : 'logging_sl_medium_50.json',
    'epochs' : 10,
    'start_epoch': start_epoch,
    'ver_start': ver_start,
    'free_size' : 50,
    'diag_size' : 75,
    'batch_size' : 2,
    'lr' : 2e-5,
    'max_len' : 100,
    'noise_type' : 'mix4',
    'device' : str(device)
}

def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if ver_start == 0 and start_epoch == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'r') as f:
        all_results = json.load(f)

# 반복문: 버전, 파라미터
# for ver in range(0, 5):
#     if ver < ver_start:
#         continue
#     print('====='*4, f'version {ver}', '====='*4)
#     set_seeds(ver)
#     meta_data['seed'] = ver

#     prepro = train_whisper_sl.CSVPreProcessor(
#         diag_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_diag_df.csv',
#         free_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_free_df.csv',
#         seed = ver)
#     train_df, valid_df = prepro.collect_data(free_df = prepro.free_df, diag_df = prepro.diag_df, free_size = meta_data['free_size'], diag_size = meta_data['diag_size'])
#     train_df = prepro.apply_text_preprocessing(train_df)
#     valid_df = prepro.apply_text_preprocessing(valid_df)

def load_prepared_data(ver):
    train_file = f'/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/csv/sl_train_df_ver_{ver}.csv'
    valid_file = f'/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/csv/sl_valid_df_ver_{ver}.csv'

    train_df = pd.read_csv(train_file)
    valid_df = pd.read_csv(valid_file)

    return train_df, valid_df

for ver in range(10):
    if ver < ver_start:
        continue
    print('====='*4, f'version {ver}', '====='*4)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    train_df, valid_df = load_prepared_data(ver)

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data': meta_data,
            'train_data': None,
            'train_data_age': None,
            'train_data_region': None,
            'train_data_sex': None,
            'time': None,
            'results': []
        }

    start = time.time()
    all_results = train_whisper_sl.train_sl(ver, meta_data, train_df, valid_df, all_results)
    all_results[str(ver)]['train_data'] = train_df['text'].tolist()
    all_results[str(ver)]['train_data_age'] = train_df['age'].tolist()
    all_results[str(ver)]['train_data_region'] = train_df['region'].tolist()
    all_results[str(ver)]['train_data_gender'] = train_df['gender'].tolist()
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)
    meta_data['start_epoch'] = 0

==================== version 0 ====================


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

CER: 19.2479, WER: 48.8402

Epoch: 10
---------------------


[C_loss : 0.0003]: 100%|██████████| 25/25 [00:18<00:00,  1.34it/s]


CER: 0.0000, WER: 0.0000


[C_loss : 0.1586]: 100%|██████████| 75/75 [00:19<00:00,  3.86it/s]


CER: 19.2479, WER: 48.8402
